<a href="https://colab.research.google.com/github/murtaza2k/AI_Residency/blob/main/rag_llamaindex_detailed_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 RAG Document Q&A Bot (Detailed Explained Version)

This notebook teaches you step-by-step how to build a **RAG (Retrieval-Augmented Generation)** system.

We will:
- Load documents
- Convert them into embeddings
- Store and reuse index
- Retrieve relevant chunks
- Generate answers using LLM
- Build a chatbot UI

---


## 📦 Step 1: Install Dependencies

In [ ]:
!pip install llama-index openai gradio python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 10.6 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: nltk
    Found existing installation: nltk 3.9.1
    Uninstalling nltk-3.9.1:
      Successfully uninstalled nltk-3.9.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, wh

## 📚 Step 2: Import Libraries (Explained)

- `os` → handles file paths and folders  
- `openai` → connects to LLM  
- `gradio` → builds chatbot UI  
- `dotenv` → loads API keys securely  

LlamaIndex components:
- `SimpleDirectoryReader` → reads documents
- `VectorStoreIndex` → creates embeddings
- `StorageContext` → manages storage
- `RetrieverQueryEngine` → RAG pipeline


In [ ]:
import os
import openai
import gradio as gr

from dotenv import load_dotenv

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    load_index_from_storage,
)
from llama_index.core.retrievers import VectorIndexRetriever
from llama_index.core.query_engine import RetrieverQueryEngine


## 🔑 Step 3: Setup API Key

We use environment variables to keep secrets safe.


In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

load_dotenv(override=True)
openai.api_key = os.getenv("OPENAI_API_KEY")

print("✅ API Key Loaded")


✅ API Key Loaded


## 📂 Step 4: Document Loader (Core RAG Logic)

This function does 2 things:

### Case 1: If index exists
- Load from storage
- Saves time and cost

### Case 2: If index does NOT exist
- Read documents from `/data`
- Convert into embeddings
- Save index for future use


In [ ]:
def document_loader():
    if os.path.exists("storage"):
        print("📦 Loading existing index...")
        storage_context = StorageContext.from_defaults(persist_dir="storage")
        index = load_index_from_storage(storage_context)
    else:
        print("🆕 Creating new index...")

        # Step 1: Read documents
        documents = SimpleDirectoryReader("data").load_data()

        # Step 2: Convert to embeddings
        index = VectorStoreIndex.from_documents(documents=documents)

        # Step 3: Save index
        index.storage_context.persist(persist_dir="storage")

    return index


## 🔍 Step 5: Query Function

This function:
1. Takes user question
2. Sends it to query engine
3. Returns answer


In [ ]:
def query_document(message, history):
    response = query_engine.query(message)
    return str(response)


## 🧠 Step 6: Retriever (Important!)

Retriever finds **most relevant chunks**

- `similarity_top_k=2` → returns top 2 matches

Example:
User asks: "Leave policy"

Retriever returns:
- Section about leave rules
- Section about exceptions


In [ ]:
index = document_loader()

retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=2
)


## ⚙️ Step 7: Query Engine

Combines:
- Retriever (search)
- LLM (generate answer)

This is the **RAG pipeline**


In [ ]:
query_engine = RetrieverQueryEngine(retriever=retriever)


## 💬 Step 8: Chat UI with Gradio

This creates a simple chatbot interface:
- Input box
- Chat history
- Response display


In [ ]:
interface = gr.ChatInterface(
    fn=query_document,
    textbox=gr.Textbox(placeholder="Ask any question from your documents!"),
    title="📄 Document RAG Bot",
    description="Ask anything from your uploaded documents"
)

interface.launch(debug=True)


## 🎯 Summary

You built:

✅ Document loader  
✅ Vector index  
✅ Retriever  
✅ Query engine  
✅ Chat UI  

---


Happy Learning 🚀
